# grayscale correlation
projecting down the results fromm 3 to 1 dimension using PCA. Then correlating the result with the individual input variables.

1. load model
2. load dataset
3. predict results using NOFE and other methods
4. project to grayscale
5. evaluate correlation

---
### Utilities

In [2]:
import sys
import json
import random
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import napari
from scipy.interpolate import griddata
from pathlib import Path

from plot_utils_era5 import *

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

sys.path.append(f"{project_root}src/")
# Import utilities
from utilities import KernelNN, KernelNN_subgraph, GraphLMDBReader, setup_device, get_output_affinities
from plot_utilities import plot_results_napari, plot_results_matplotlib

sys.path.append(f"{project_root}/ERA5/scripts/")
from utilities_era5 import ERA5_Graph_Generator as Data_Generator
from utilities_era5 import ERA5_Graph_Generator_perSampleNorm as Data_Generator_old
data_path = project_root / 'ERA5' / 'data'

# Device setup
device = setup_device()
print(f'Using device: {device}')

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import subprocess

def load_model(id:str, model_type:str='random_crop'):
    model_path = project_root / 'ERA5' / 'trained_models' / model_type / f'{id}.pth'
    config_path = project_root / 'ERA5' / 'trained_models' / model_type / 'logs' / 'configs' / f'{id}.json' 
    
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    model = KernelNN(
        width=config['width'],
        ker_width=config['ker_width'],
        depth=config['depth'],
        ker_in=config['edge_attr'],
        in_width=config['node_features'],
        out_width=config['out_width']
    ).to(device)
    
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    return model, config

using Device: mps
Using device: mps


----
### load model + dataset 

In [3]:
#model, config = load_model(id='model5_grayscale_2026-04-28_t12-23-06' , model_type='regular')
model, config = load_model(id='test1_model5_median_tkernel_2026-04-22_t21-22-37' , model_type='regular')

In [4]:
lat_min = config["lat_min"]
lat_max = config["lat_max"]
lon_min = config["lon_min"]
lon_max = config["lon_max"]
#N = config["N"]
k = config["k"]
perplexity = config["perplexity"]
sampling_method = config["sampling_method"]
train_date_start_end = config["train_date_start_end"]
test_date_start_end = config["test_date_start_end"]
train_samples_per_date = config["train_samples_per_date"]
test_samples_per_date = config["test_samples_per_date"]
feature_mode = config["feature_mode"]
vars = config["vars"]
sigma_method = config["sigma_method"]
output_dim = config["out_width"]

In [32]:
date = "2019-01-10"

bbox = lon_min, lat_min, lon_max, lat_max

#Generator = Data_Generator_old(data_path)
Generator = Data_Generator(data_path)
Generator.load_data(bbox=bbox, vars=vars, filename=f'{date}.nc')
print(f'total points available: {Generator.da.shape[1]*Generator.da.shape[2]}')

N=10000
print('generate data')
graph = Generator.sample_graph(N=N, k=k, perplexity=perplexity, sampling_method=sampling_method, gen_feature_mode=feature_mode, sigma_method=sigma_method)
sample = graph.to(device)

locations = pd.DataFrame({"lat": sample.meta.gps[:,0], "lon": sample.meta.gps[:,1]})
X = sample.normed_values.detach().cpu().numpy()

total points available: 32761
generate data


---
## Get Predictions

In [38]:
# NOFE
print('NOFE ...', end='\r')
pred_nofe = model(sample).detach().cpu().numpy()
print('NOFE\t- done')

# PCA
print('PCA ...', end='\r')
pca = PCA(n_components=output_dim)
pred_pca = pca.fit_transform(X)
print('PCA\t- done')

# t-SNE
print('t-SNE ...', end='\r')
pred_tsne = TSNE(
    n_components=output_dim,
    perplexity=30,
    learning_rate="auto",
    random_state=42
).fit_transform(X)
print('t-SNE\t- done')

# UMAP
print('UMAP ...', end='\r')
umap_id = 'umap_buffer'
dir = f'{project_root}/ERA5/umap'
np.save(f'{dir}/{umap_id}.npy', X)
subprocess.run(f"cd {dir} && conda run -n umap python execute_umap.py {umap_id}", shell=True)
pred_umap = np.load(f'{dir}/res_{umap_id}.npy')

os.remove(f'{dir}/{umap_id}.npy')
os.remove(f'{dir}/res_{umap_id}.npy')
print('UMAP\t- done')

NOFE	- done
PCA	- done
t-SNE	- done
importing libraries
reading in data
applying umap
saving file
"
file saved as res_umap_buffer.npy"

UMAP	- done


/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")



---
## Grayscale Projection

In [34]:
pca = PCA(n_components=1)
grayscale_nofe = pca.fit_transform(pred_nofe).squeeze() 
grayscale_pca = pca.fit_transform(pred_pca).squeeze() 
grayscale_tsne = pca.fit_transform(pred_tsne).squeeze() 
grayscale_umap = pca.fit_transform(pred_umap).squeeze() 

---
## Correlate

In [35]:
correlations_nofe = np.array([
    np.abs(np.corrcoef(grayscale_nofe, X[:, i])[0, 1])
    for i in range(X.shape[1])
])

correlations_pca = np.array([
    np.abs(np.corrcoef(grayscale_pca, X[:, i])[0, 1])
    for i in range(X.shape[1])
])

correlations_tsne = np.array([
    np.abs(np.corrcoef(grayscale_tsne, X[:, i])[0, 1])
    for i in range(X.shape[1])
])

correlations_umap = np.array([
    np.abs(np.corrcoef(grayscale_umap, X[:, i])[0, 1])
    for i in range(X.shape[1])
])

In [36]:
corrs = pd.DataFrame({
    'nofe' : correlations_nofe,
    'pca' : correlations_pca,
    'tsne' : correlations_tsne,
    'umap' : correlations_umap
}, index=config['vars'])
print(corrs)
print()
print('Mean Values:')
print(f'NOFE:\t{corrs.nofe.mean():.2f}')
print(f'PCA:\t{corrs.pca.mean():.2f}')
print(f't-SNE:\t{corrs.tsne.mean():.2f}')
print(f'UMAP:\t{corrs.umap.mean():.2f}')

        nofe       pca      tsne      umap
w   0.108967  0.208199  0.096902  0.168564
v   0.107447  0.120404  0.323904  0.122951
z   0.403067  0.531306  0.650070  0.222298
q   0.254267  0.769116  0.173650  0.628648
r   0.218310  0.482880  0.294682  0.126813
t   0.295162  0.646618  0.780129  0.704732
d   0.499210  0.104731  0.023646  0.006215
o3  0.620984  0.735376  0.024640  0.102569
pv  0.685889  0.686621  0.180353  0.063894
u   0.015379  0.138070  0.392127  0.275977
vo  0.642161  0.823079  0.099929  0.333487

Mean Values:
NOFE:	0.35
PCA:	0.48
t-SNE:	0.28
UMAP:	0.25


---
# Run on full scale datset

In [5]:
config['dataset_id']

'dataset_reg_fixcrop_median_woPos_tkernel'

In [6]:
dates = pd.date_range("2019-01-01", "2019-01-31").strftime("%Y-%m-%d").tolist()

#Generator = Data_Generator_old(data_path)
Generator = Data_Generator(data_path)

bbox = lon_min, lat_min, lon_max, lat_max


print('generate data')

graphs=[]
locations_list=[]
X_list=[]

for date in dates:
    N=random.randint(1000,10000)
    Generator.load_data(bbox=bbox, vars=vars, filename=f'{date}.nc')
    print(f'total points available: {Generator.da.shape[1]*Generator.da.shape[2]}')
    graph = Generator.sample_graph(N=N, k=k, perplexity=perplexity, sampling_method=sampling_method, gen_feature_mode=feature_mode, sigma_method='ones')
    sample = graph.to(device)
    graphs.append(sample)
    locations_list.append(pd.DataFrame({"lat": sample.meta.gps[:,0], "lon": sample.meta.gps[:,1]}))
    X_list.append(sample.normed_values.detach().cpu().numpy())

generate data
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761


In [7]:
#dataset = GraphLMDBReader(f'{project_root}/ERA5/datasets/regular/{config["dataset_id"]}_test.lmdb')
dataset = graphs
print(f'loaded {len(dataset)} samples')

loaded 31 samples


In [8]:
corr_res_nofe = []

#for s in range(len(reader)):
for s in range(len(dataset)):
    print(f'{s+1}/{len(dataset)}', end='\r')

    sample = dataset[s].to(device='mps')
    X = sample.normed_values.detach().cpu().numpy()
    # =================== 
    # NOFE
    # =================== 
    pred_nofe = model(sample).detach().cpu().numpy()
    pca = PCA(n_components=1)
    grayscale_nofe = pca.fit_transform(pred_nofe).squeeze() 
    correlations_nofe = np.array([
        np.abs(np.corrcoef(grayscale_nofe, X[:, i])[0, 1])
        for i in range(X.shape[1])
    ])
    corr_res_nofe.append(correlations_nofe)

corr_res_nofe = np.vstack(corr_res_nofe)
print(corr_res_nofe.mean())

0.43042566233785545


In [10]:
corr_res_pca = []

#for s in range(len(reader)):
for s in range(len(dataset)):
    print(f'{s+1}/{len(dataset)}', end='\r')

    sample = dataset[s].to(device='mps')
    X = sample.normed_values.detach().cpu().numpy()
    # =================== 
    # PCA
    # =================== 
    pca = PCA(n_components=3)
    pred_pca = pca.fit_transform(X)
    pca = PCA(n_components=1)
    grayscale_pca = pca.fit_transform(pred_pca).squeeze() 
    correlations_pca = np.array([
        np.abs(np.corrcoef(grayscale_pca, X[:, i])[0, 1])
        for i in range(X.shape[1])
    ])
    corr_res_pca.append(correlations_pca)

corr_res_pca = np.vstack(corr_res_pca)    

In [11]:
corr_res_tsne = []

#for s in range(len(reader)):
for s in range(len(dataset)):
    print(f'{s+1}/{len(dataset)}', end='\r')

    sample = dataset[s].to(device='mps')
    X = sample.normed_values.detach().cpu().numpy()
    # =================== 
    # t-SNE
    # =================== 
    pred_tsne = TSNE(
        n_components=3,
        perplexity=30,
        learning_rate="auto",
        random_state=42
    ).fit_transform(X)
    pca = PCA(n_components=1)
    grayscale_tsne = pca.fit_transform(pred_tsne).squeeze() 
    correlations_tsne = np.array([
        np.abs(np.corrcoef(grayscale_tsne, X[:, i])[0, 1])
        for i in range(X.shape[1])
    ])
    corr_res_tsne.append(correlations_tsne)

corr_res_tsne = np.vstack(corr_res_tsne)

python(25627) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [12]:
corr_res_umap = []

#for s in range(len(reader)):
for s in range(len(dataset)):
    print(f'{s+1}/{len(dataset)}', end='\r')

    sample = dataset[s].to(device='mps')
    X = sample.normed_values.detach().cpu().numpy()
    # =================== 
    # UMAP
    # =================== 
    umap_id = 'umap_buffer'
    dir = f'{project_root}/ERA5/umap'
    np.save(f'{dir}/{umap_id}.npy', X)
    subprocess.run(f"cd {dir} && conda run -n umap python execute_umap.py {umap_id}", shell=True)
    pred_umap = np.load(f'{dir}/res_{umap_id}.npy')
    os.remove(f'{dir}/{umap_id}.npy')
    os.remove(f'{dir}/res_{umap_id}.npy')
    pca = PCA(n_components=1)
    grayscale_umap = pca.fit_transform(pred_umap).squeeze() 
    correlations_umap = np.array([
        np.abs(np.corrcoef(grayscale_umap, X[:, i])[0, 1])
        for i in range(X.shape[1])
    ])
    corr_res_umap.append(correlations_umap)

corr_res_umap = np.vstack(corr_res_umap)

python(40363) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")

python(40957) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


importing libraries
reading in data
applying umap
saving file
"
file saved as res_umap_buffer.npy"

importing libraries
reading in data
applying umap
saving file
"
file saved as res_umap_buffer.npy"



/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")

python(41432) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


importing libraries
reading in data
applying umap
saving file
"
file saved as res_umap_buffer.npy"



/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")

python(41746) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as ea

importing libraries
reading in data
applying umap
saving file
"
file saved as res_umap_buffer.npy"



... (39 outputs left out)

importing libraries
reading in data
applying umap
saving file
"
file saved as res_umap_buffer.npy"

importing libraries
reading in data
applying umap
saving file
"
file saved as res_umap_buffer.npy"



/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")

python(50804) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


importing libraries
reading in data
applying umap
saving file
"
file saved as res_umap_buffer.npy"



/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")



In [21]:
corrs = pd.DataFrame({
    'nofe' : corr_res_nofe.mean(axis=0),
    'pca' : corr_res_pca.mean(axis=0),
    'tsne' : corr_res_tsne.mean(axis=0),
    'umap' : corr_res_umap.mean(axis=0)
}, index=config['vars'])

print('Mean correlation per variable:')
print(corrs)

print('\nMean correlation coefficient:\n')
print(f'NOFE:\t{corr_res_nofe.mean():.3f}')
print(f'PCA:\t{corr_res_pca.mean():.3f}')
print(f't-SNE:\t{corr_res_tsne.mean():.3f}')
print(f'UMAP:\t{corr_res_umap.mean():.3f}')

Mean correlation per variable:
        nofe       pca      tsne      umap
w   0.419036  0.124227  0.074207  0.083346
v   0.180423  0.277856  0.213890  0.233617
z   0.229263  0.799658  0.891768  0.865899
q   0.727376  0.690996  0.503240  0.472464
r   0.649697  0.204876  0.290615  0.286497
t   0.277066  0.830525  0.886848  0.856446
d   0.162892  0.059765  0.037509  0.027374
o3  0.578982  0.334538  0.117711  0.140248
pv  0.698662  0.645929  0.282865  0.282115
u   0.109541  0.303697  0.273586  0.294415
vo  0.701744  0.589227  0.181188  0.188166

Mean correlation coefficient:

NOFE:	0.430
PCA:	0.442
t-SNE:	0.341
UMAP:	0.339


In [20]:
for i in corrs['umap']:
    print(f'{round(i,3)}, & ', end ='')

0.083, & 0.234, & 0.866, & 0.472, & 0.286, & 0.856, & 0.027, & 0.14, & 0.282, & 0.294, & 0.188, & 

In [22]:
corrs.shape

(11, 4)

In [26]:
for i in range(corrs.shape[0]):
    print(f"{round(corrs['nofe'][i],3)} & {round(corrs['pca'][i],3)} & {round(corrs['tsne'][i],3)} & {round(corrs['umap'][i],3)}")

0.419 & 0.124 & 0.074 & 0.083
0.18 & 0.278 & 0.214 & 0.234
0.229 & 0.8 & 0.892 & 0.866
0.727 & 0.691 & 0.503 & 0.472
0.65 & 0.205 & 0.291 & 0.286
0.277 & 0.831 & 0.887 & 0.856
0.163 & 0.06 & 0.038 & 0.027
0.579 & 0.335 & 0.118 & 0.14
0.699 & 0.646 & 0.283 & 0.282
0.11 & 0.304 & 0.274 & 0.294
0.702 & 0.589 & 0.181 & 0.188


/var/folders/sw/tjsbk4pj7q9gs_91lbj2_2rc0000gp/T/ipykernel_4194/2999066100.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"{round(corrs['nofe'][i],3)} & {round(corrs['pca'][i],3)} & {round(corrs['tsne'][i],3)} & {round(corrs['umap'][i],3)}")
